In [8]:
import xarray as xr
from scipy import io
import numpy as np
import pandas as pd
import seawater as sw
import matplotlib.pyplot as plt
import cmocean
import gsw
#this is just a useful function        
def moving_average(x,n, window = "flat"):
    if n%2 == 0:
        n+=1
    N = x.size
    cx = np.full(x.size, np.nan)
    for i in range(N):
        ii = np.arange(i-n//2, i+n//2+1,1)
        if window == "flat":
            ww = np.ones(ii.size)
        elif window == "gauss":
            xx = ii - i
            
            ww = np.exp(- xx**2/(float(n)/4)**2 )
        elif window == "hanning":
            ww = np.hanning(ii.size)
        ww = ww[ (ii>=0) & (ii<N)]
        ii = ii[ (ii>=0) & (ii<N)]
        
        #print(ii)
        kk = np.isfinite(x[ii])
        if np.sum(kk)<0.25*ii.size:
            continue
        cx[i] = np.sum(x[ii[kk]]*ww[kk])/np.sum(ww[kk])
    return cx

AttributeError: partially initialized module 'pandas' from 'c:\Users\Manuelito\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

# First, load datafiles

In [ ]:
dsvmp = io.loadmat("OVIDE2008_processed_VMP6000_final.mat")
dsctd = io.loadmat("OVIDE_VMP_CTD_merged.mat")

#for all variables it only gets data from depth 3 to avoid noise close to the surface
lon_vmp = dsvmp["longitude"][0,:]
lat_vmp = dsvmp["latitude"][0,:]

pres_vmp = dsvmp["pres"][0,3:].astype(float)
epsilon = dsvmp["epsilon"][3:,:]
chi = dsvmp["chi"][3:,:]

grT = dsvmp["grT"][3:,:]
grS = dsvmp["grS"][3:,:]
theta = dsvmp["theta"][3:,:]
salinity = dsvmp["S"][3:,:]
T = dsvmp["T"][3:,:]
gamman = dsvmp["gamman"][3:,:]#dsvmp["gamman"][3:,:]

depth = np.full(theta.shape, np.nan)
for i in range(lon_vmp.size):
    for j in range(pres_vmp.size):
        depth[j,i] = sw.dpth(pres_vmp[j], lat_vmp[i])
        
print(min(lat_vmp))
print(max(lat_vmp))
print(min(lon_vmp))
print(max(lon_vmp))

In [ ]:
#replaces first ctd profile which was broken (first VMP station was in station 8 -  so index 7)
T[:,0] = dsctd["temperature"][:,7]
salinity[:,0] = dsctd["salinity"][:,7]
gamman[:,0] = dsctd["gamman"][:,7]
theta[:,0] = sw.ptmp(salinity[:,0], T[:,0], pres_vmp,0)

#calculates the vertical gradient
for j in range(1,pres_vmp.size-1):
    grT[j,0] = -(theta[j+1,0] - theta[j-1,0])/(depth[j+1,0] - depth[j-1,0])
    grS[j,0] = -(salinity[j+1,0] - salinity[j-1,0])/(depth[j+1,0] - depth[j-1,0])
    
land = np.isnan(dsctd["temperature"])
plt.pcolor(land)

In [ ]:
"""
ii = np.where(np.isfinite(dsctd["epsilon"][20,:]))[0]

for i,j in enumerate(ii):
    T[:,i] = dsctd["temperature"][:,j]
    salinity[:,i] = dsctd["salinity"][:,j]
    gamman[:,i] = dsctd["gamman"][:,j]
    theta[:,i] = sw.ptmp(salinity[:,i], T[:,i], pres_vmp,0)

#calculates the vertical gradient
for i in range(epsilon.shape[1]):
    for j in range(1,pres_vmp.size-1):
        grT[j,i] = -(theta[j+1,i] - theta[j-1,i])/(depth[j+1,i] - depth[j-1,i])
        grS[j,i] = -(salinity[j+1,i] - salinity[j-1,i])/(depth[j+1,i] - depth[j-1,i])
"""

In [ ]:
oxygen = np.full(T.shape, np.nan)
ii = np.where(np.isfinite(np.nanmean(dsctd["epsilon"], axis = 0)))[0]
print(ii)
for i,j in enumerate(ii):
    oxygen[:,i] = dsctd["oxygen"][:,j]
    
grO = np.full(T.shape, np.nan)
for i in range(epsilon.shape[1]):
    for j in range(1,pres_vmp.size-1):
        grO[j,i] = -(oxygen[j+1,i] - oxygen[j-1,i])/(depth[j+1,i] - depth[j-1,i])
   

In [ ]:
#TEOS 10
pres2d = np.tile(pres_vmp, (lon_vmp.size,1)).T
Lon2d = np.tile(lon_vmp, (pres_vmp.size,1))
Lat2d = np.tile(lat_vmp, (pres_vmp.size,1))
print(Lon2d.shape)
SA = gsw.SA_from_SP(salinity,pres2d,Lon2d,Lat2d)
CT = gsw.CT_from_pt(SA,theta)
sigma0 = gsw.sigma0(SA, CT)+1000.
sp = gsw.spiciness0(SA,CT)

In [ ]:
gamman0 = np.copy(gamman)
#gamman = neutral-1000

## make section plot

In [ ]:
fig, ax = plt.subplots(2,1,sharex = True, sharey = True, figsize = (9,9))
cc=ax[0].pcolor(lon_vmp, pres_vmp, np.log10(epsilon), vmin = -11, vmax = -8)
plt.colorbar(cc, ax = ax[0])
ax[0].set_title("$\\varepsilon$ (W/kg)")

cc=ax[1].pcolor(lon_vmp, pres_vmp, np.log10(chi), vmin = -12, vmax = -7)
plt.colorbar(cc, ax = ax[1])
ax[0].set_ylim((6010,0))
ax[1].set_title("$\\chi_{\\theta}$ (K$^2$/s)")

# Sorts the density profiles (from low density to high density to remove overturns)

In [ ]:

gamman_s = np.full(theta.shape, np.nan)
theta_s = np.full(theta.shape, np.nan)
salinity_s = np.full(salinity.shape, np.nan)
oxygen_s = np.full(salinity.shape, np.nan)
for i in range(lon_vmp.size):
    sg0 = gamman[:,i]
    iif = np.isfinite(sg0)
    gamman_s[iif,i] = np.sort(gamman[iif,i])
    iis = np.argsort(gamman[iif,i])
    theta_s[iif,i] =theta[iif,i][iis]
    salinity_s[iif,i] = salinity[iif,i][iis]
    oxygen_s[iif,i] = oxygen[iif,i][iis]


# Builds the smooth "mean flow" temperature and salinity profiles

In [ ]:
h = 400 #defines the segment length
theta_mean = np.full(theta.shape, np.nan)
salinity_mean = np.full(theta.shape, np.nan)
oxygen_mean = np.full(theta.shape, np.nan)
for i in range(lon_vmp.size):
    print(i)
    #sg_smooth = moving_average(gamman_s[:,i],10)
    for j in range(pres_vmp.size):
        deepest = np.where(np.isfinite(theta[:,i]))[0][-1]
        deepest_pres = pres_vmp[deepest]
        min_pres = pres_vmp[j]-h/2
        max_pres = pres_vmp[j]+h/2
        if min_pres<0:
            
            max_pres+=np.abs(min_pres)
            min_pres= 0
        if max_pres>deepest_pres:
            delta = max_pres-deepest_pres
            min_pres-=delta
            max_pres = np.copy(deepest_pres)
            
        jj = np.where( (pres_vmp>=min_pres) & (pres_vmp<=max_pres) & (np.isfinite(gamman_s[:,i])))[0]
        
        pt = np.polyfit(gamman_s[jj,i],theta_s[jj,i],4)
        ppt = np.poly1d(pt)
        theta_mean[j,i] = ppt(gamman_s[j,i])

        
        ps = np.polyfit(gamman_s[jj,i],salinity_s[jj,i],4)
        pps = np.poly1d(ps)
        salinity_mean[j,i] = pps(gamman_s[j,i])
        
        po = np.polyfit(gamman_s[jj,i],oxygen_s[jj,i],4)
        ppo = np.poly1d(po)
        oxygen_mean[j,i] = ppo(gamman_s[j,i])
        

        

# Calculates the "mean flow" temperature, salinity gradients and N2

In [ ]:
grT_mean = np.full(theta.shape, np.nan)
grS_mean = np.full(theta.shape, np.nan)
grO_mean = np.full(theta.shape, np.nan)
N2 = np.full(theta.shape, np.nan)
for i in range(lon_vmp.size):
    
    for j in range(1,pres_vmp.size-1):
        grT_mean[j,i] = -(theta_mean[j+1,i] - theta_mean[j-1,i])/(depth[j+1,i] - depth[j-1,i])
        grS_mean[j,i] = -(salinity_mean[j+1,i] - salinity_mean[j-1,i])/(depth[j+1,i] - depth[j-1,i])
        grO_mean[j,i] = -(oxygen_mean[j+1,i] - oxygen_mean[j-1,i])/(depth[j+1,i] - depth[j-1,i])
        
        N2[j,i] = 9.81/1027*(gamman_s[j+1,i] -gamman_s[j-1,i])/(depth[j+1,i] - depth[j-1,i])

N2[N2<1e-8] = np.nan #removes very low N2 in the bottom boundary layer, which pose problems

# Caculates terms of the salinity and temperature variance budget

In [ ]:
Krho = 0.2*epsilon/N2 #diapycnal diffusivity

chi_dia = 2*Krho*grT**2 #diapycnal production of thermal variance including fine scale: should be similar to chi

Pdia=  2*Krho*grT_mean**2 #diapycnal production of thermal on the mean flow profile

chiS = 2*Krho*grS**2 #diapycnal production of haline variance including fine scale: should be similar to chi

PdiaS=  2*Krho*grS_mean**2  #diapycnal production of haline on the mean flow profile

chiS[chiS>1e-4] = np.nan #removes some outlier

chiO = 2*Krho*grO**2 #diapycnal production of haline variance including fine scale: should be similar to chi

PdiaO=  2*Krho*grO_mean**2  #diapycnal production of haline on the mean flow profile

## tests that local diapycnal production roughly balances chi (like figure 2 in my paper)

In [ ]:
import cmocean 

dchi = 0.25
xchi = np.arange(-14.5,-1.5,dchi)
hist_chis = np.full((xchi.size, xchi.size), 0)
hist_chi_dia = np.full((xchi.size, xchi.size), 0)

for i in range(xchi.size):
    ii0 = (np.abs( np.log10(chi_dia)-xchi[i] )<dchi/2)
    hist_chi_dia[i] = np.sum(ii0)
    for j in range(xchi.size):
        ii = (np.abs( np.log10(chi)-xchi[i] )<dchi/2) & (np.abs( np.log10(chi_dia)-xchi[j] )<dchi/2)
        hist_chis[i,j] = np.sum(ii )

fig, ax = plt.subplots()
#ax.loglog(chi.ravel(), chi_dia0.ravel(),'.')
cc=ax.contourf(xchi,xchi, hist_chis,50, cmap  = cmocean.cm.rain)
cb = plt.colorbar(cc)
#cb.set_ticks(np.arange(0,4,1))
#cb.set_ticklabels( 10**np.arange(0,4,1))
ax.plot([-14,-3],[-14,-3],color = "k")
ax.plot([-14,-3],[-14+np.log10(10),-3+np.log10(10)],color = "k", ls = "dotted",lw = 1)
ax.plot([-14,-3],[-14-np.log10(10),-3-np.log10(10)],color = "k", ls = "dotted",lw = 1)
ax.plot([-14,-3],[-14+np.log10(2),-3+np.log10(2)],color = "k", ls = "--",lw = 1)
ax.plot([-14,-3],[-14-np.log10(2),-3-np.log10(2)],color = "k", ls = "--",lw = 1)

ax.annotate("10:1", xy = (-7,-6), ha = "center", va = "bottom", rotation = 45, fontsize = 8)
ax.annotate("2:1", xy = (np.log10(2e-7),np.log10(4e-7)), ha = "center", va = "bottom", rotation = 45, fontsize = 8)

ax.annotate("1:10", xy = (np.log10(1.2e-6),np.log10(1.2e-7)), ha = "center", va = "top", rotation = 45, fontsize = 8)
ax.annotate("1:2", xy = (np.log10(6e-7),np.log10(3e-7)), ha = "center", va = "top", rotation = 45, fontsize = 8)

ax.set_xlim((-14,-5))
ax.set_ylim((-14,-5))
ax.set_xticks(np.arange(-14,-4))
ax.set_xticklabels(["$10^{%s}$"%(x) for x in range(-14,-4)])
ax.set_yticks(np.arange(-14,-4))
ax.set_yticklabels(["$10^{%s}$"%(x) for x in range(-14,-4)])
ax.set_xlabel(" $ P_{\\theta^2} = 2 K_{\\rho} (\partial _{z} \\overline{\\theta})^2$ (K$^2$/s)")
ax.set_ylabel(" $\\chi_{\\theta}$ (K$^2$/s)")
ax.grid(True)
fig.savefig("chi_vs_localP.png", dpi =300, bbox_inches = "tight")

fig, ax = plt.subplots(1)
ax.plot(xchi, np.sum(hist_chis, axis = 1))
ax.plot(xchi, np.sum(hist_chis, axis = 0))


# Makes some plots 

## cruise mean variance budget profiles

In [ ]:
fig, ax = plt.subplots(1,2, sharey = True, figsize = (9,4))


ax[0].semilogx(np.nanmean(Pdia[:,1:], axis =1), pres_vmp,label ="$P_{\\theta}^{\\perp}$")
ax[0].semilogx(np.nanmean(chi_dia[:,1:],axis =1), pres_vmp, color = "gray", label ="$P_{\\theta}^{\\perp}$" )
ax[0].semilogx(np.nanmean(chi[:,1:], axis =1),  pres_vmp, color = "k", label = "$\chi_{\\theta}$")
ax[0].set_ylim((4000,0))
ax[0].set_xlim((5e-12,1e-6))
ax[0].legend()
ax[0].set_ylabel("Pressure (dbar)")
ax[0].set_title("Thermal variance budget")
ax[0].set_xlabel("K$^2$/s")

ax[1].semilogx(np.nanmean(PdiaS,axis =1), pres_vmp,label ="$P_S^{\\perp}$")
ax[1].semilogx(np.nanmean(chiS,axis =1), pres_vmp, color = "gray", label = "$P_S \\approx \chi_S$")
ax[1].legend()
ax[1].set_title("Haline variance budget")
ax[1].set_xlabel("PSU$^2$/s")

ax[1].set_ylim((4000,0))
ax[1].set_xlim((1e-14,1e-6))



In [ ]:
fig, ax = plt.subplots(1,1, sharey = True, figsize = (4,4))


ax.semilogx(np.nanmean(PdiaO[:,1:], axis =1), pres_vmp,label ="$P_{\\theta}^{\\perp}$")
#ax[0].semilogx(np.nanmean(chi_dia[:,1:],axis =1), pres_vmp, color = "gray", label ="$P_{\\theta}^{\\perp}$" )
ax.semilogx(np.nanmean(chiO[:,1:], axis =1),  pres_vmp, color = "k", label = "$\chi_{\\theta}$")
ax.set_ylim((4000,0))
ax.set_xlim((5e-10,1e-4))
ax.legend()
ax.set_ylabel("Pressure (dbar)")
ax.set_title("Oxygen variance budget")
ax.set_xlabel("(umol/kg)$^2$/s")

In [ ]:
fig, ax = plt.subplots(1,3, sharey = True, figsize = (12,4))


ax[0].semilogx(moving_average(np.nanmean(Pdia[:,:], axis =1),30,window = "gauss"), pres_vmp,label ="$P_{\\theta}^{\\perp}$")
ax[0].semilogx(moving_average(np.nanmean(chi_dia[:,:],axis =1),30,window = "gauss"), pres_vmp, color = "gray", label ="$P_{\\theta}^{\\perp}$" )
ax[0].semilogx(moving_average(np.nanmean(chi[:,:], axis =1),30,window = "gauss"),  pres_vmp, color = "k", label = "$\chi_{\\theta}$")
ax[0].set_ylim((4000,0))
ax[0].set_xlim((5e-12,1e-6))
ax[0].legend()
ax[0].set_ylabel("Pressure (dbar)")
ax[0].set_title("Thermal variance budget")
ax[0].set_xlabel("K$^2$/s")
ax[0].grid(True)

ax[1].semilogx(moving_average(np.nanmean(PdiaS,axis =1),30,window = "gauss"), pres_vmp,label ="$P_S^{\\perp}$")
ax[1].semilogx(moving_average(np.nanmean(chiS,axis =1),30,window = "gauss"), pres_vmp, color = "gray", label = "$P_S \\approx \chi_S$")
ax[1].legend()
ax[1].set_title("Haline variance budget")
ax[1].set_xlabel("PSU$^2$/s")
ax[1].grid(True)
ax[1].set_ylim((4000,0))
ax[1].set_xlim((1e-13,1e-6))

ax[2].semilogx(moving_average(np.nanmean(PdiaO,axis =1),30,window = "gauss"), pres_vmp,label ="$P_O^{\\perp}$")
ax[2].semilogx(moving_average(np.nanmean(chiO,axis =1),30,window = "gauss"), pres_vmp, color = "gray", label = "$P_O \\approx \chi_O$")
ax[2].legend()
ax[2].set_title("Oxygen variance budget")
ax[2].set_xlabel("(umol/kg))$^2$/s")

ax[2].set_ylim((4000,0))
ax[2].set_xlim((1e-8,1e-3))

In [ ]:
dschi0 = xr.open_dataset("/home/bieito/Documents/SCIENCE/isopycnal_climatology_analysis/variance_dissipation_and_fluxes_WOAgrid_0.1sigma.nc")
#dschi0 = dschi0.interp1()
#display(dschi0)
dschi = dschi0.interp( Lon = xr.DataArray(lon_vmp), Lat = xr.DataArray(lat_vmp) , method = "nearest")

In [ ]:

clim_chiCT_tot_IrmS = np.nanmean(dschi.chiCT_dia[:,16:]+dschi.chiCT_iso[:,16:], axis =1)

clim_chiSA_tot_IrmS = np.nanmean(dschi.chiSA_dia[:,16:]+dschi.chiSA_iso[:,16:], axis =1)

clim_chiCT_dia_IrmS = np.nanmean(dschi.chiCT_dia[:,0:16], axis =1)

clim_chiSA_dia_IrmS = np.nanmean(dschi.chiSA_dia[:,0:16], axis =1)

#clim_Z_WEB = np.nanmean(dschi.z[:,AREASst==1], axis = 1)
clim_Z_IrmS = np.nanmean(dschi.z[:,16:], axis = 1)

fig, ax = plt.subplots(1,3, sharey = True, figsize = (12,4))


ax[0].semilogx(moving_average(np.nanmean(Pdia[:,16:], axis =1),30,window = "gauss"), pres_vmp,label ="$P_{\\theta}^{\\perp}$")
ax[0].semilogx(moving_average(np.nanmean(chi_dia[:,16:],axis =1),30,window = "gauss"), pres_vmp, color = "gray", label ="$P_{\\theta}^{\\perp}$" )
ax[0].semilogx(moving_average(np.nanmean(chi[:,16:], axis =1),30,window = "gauss"),  pres_vmp, color = "k", label = "$\chi_{\\theta}$")
ax[0].semilogx(clim_chiCT_tot_IrmS, clim_Z_IrmS, "o", mec = "k", mfc = "w")
ax[0].semilogx(clim_chiCT_dia_IrmS, clim_Z_IrmS, "o", mec = "b", mfc = "w")

ax[0].set_ylim((4000,0))
ax[0].set_xlim((5e-12,1e-6))
ax[0].legend()
ax[0].set_ylabel("Pressure (dbar)")
ax[0].set_title("Thermal variance budget")
ax[0].set_xlabel("K$^2$/s")
ax[0].grid(True)

ax[1].semilogx(moving_average(np.nanmean(PdiaS[:,16:],axis =1),30,window = "gauss"), pres_vmp,label ="$P_S^{\\perp}$")
ax[1].semilogx(moving_average(np.nanmean(chiS[:,16:],axis =1),30,window = "gauss"), pres_vmp, color = "gray", label = "$P_S \\approx \chi_S$")
#ax[1].semilogx(moving_average(np.nanmean(chiS[:,0:11],axis =1),30,window = "gauss"), pres_vmp, color = "gray", label = "$P_S \\approx \chi_S$")

ax[1].semilogx(clim_chiSA_tot_IrmS, clim_Z_IrmS, "o", mec = "k", mfc = "w")
ax[1].semilogx(clim_chiSA_dia_IrmS, clim_Z_IrmS, "o", mec = "b", mfc = "w")

ax[1].legend()
ax[1].set_title("Haline variance budget")
ax[1].set_xlabel("PSU$^2$/s")
ax[1].grid(True)
ax[1].set_ylim((4000,0))
ax[1].set_xlim((1e-14,1e-6))

ax[2].semilogx(moving_average(np.nanmean(PdiaO[:,16:],axis =1),30,window = "gauss"), pres_vmp,label ="$P_O^{\\perp}$")
ax[2].semilogx(moving_average(np.nanmean(chiO[:,16:],axis =1),30,window = "gauss"), pres_vmp, color = "gray", label = "$P_O \\approx \chi_O$")
ax[2].legend()
ax[2].set_title("Oxygen variance budget")
ax[2].set_xlabel("(umol/kg))$^2$/s")

ax[2].set_ylim((4000,0))
ax[2].set_xlim((1e-9,1e-3))

ax[1].set_title("Irminger")

# diapycnal contribution to mixing sections

In [ ]:
#smooth fields
Ns = 50 #25
print(pres_vmp)
chi_smooth = np.full(chi.shape, np.nan)
Pdia_smooth = np.full(chi.shape, np.nan)
chiS_smooth = np.full(chi.shape, np.nan)
PdiaS_smooth = np.full(chi.shape, np.nan)
chiO_smooth = np.full(chi.shape, np.nan)
PdiaO_smooth = np.full(chi.shape, np.nan)
for i in range(lon_vmp.size):
    chi_smooth[:,i] = moving_average(chi[:,i],Ns)
    Pdia_smooth[:,i] = moving_average(Pdia[:,i],Ns)
    
    chiS_smooth[:,i] = moving_average(chiS[:,i],Ns)
    PdiaS_smooth[:,i] = moving_average(PdiaS[:,i],Ns)
    
    chiO_smooth[:,i] = moving_average(chiO[:,i],Ns)
    PdiaO_smooth[:,i] = moving_average(PdiaO[:,i],Ns)

plt.plot( lon_vmp,np.nanmedian( (Pdia/chi)[:96,:], axis = 0)*100,".-" ) 
plt.plot( lon_vmp,np.nanmedian( (PdiaS/chiS)[:96,:], axis = 0)*100,".-" ) 
plt.ylim((0,100))

In [ ]:
#smooth fields
display(dsctd.keys())

lon2d_ctd = np.tile(dsctd["longitude"],(dsctd["pressure"].size,1))
lat2d_ctd = np.tile(dsctd["latitude"],(dsctd["pressure"].size,1))

SA_ctd = gsw.SA_from_SP(dsctd["salinity"],dsctd["pressure_2d"],lon2d_ctd,lat2d_ctd)
CT_ctd = gsw.CT_from_t(SA_ctd,dsctd["temperature"],dsctd["pressure_2d"])

nst = dsctd["longitude"].size


In [ ]:

b= 0.7
zlabels = np.array([0, 500, 1500, 3000, 6000])

bottom = np.full(dsctd["longitude"][0,:].size, np.nan)
for i in range(dsctd["longitude"][0,:].size):
    ij = np.where(np.isfinite(dsctd["temperature"][:,i]))[0][-1]
    bottom[i] = dsctd["pressure"][0,ij]
print(bottom)


long_ctd = dsctd["longitude"][0,:]
ist = np.argsort(long_ctd)[:-3]
long_ctd = long_ctd[ist][:-3]
    
fig, ax = plt.subplots(2,2,sharex = True, sharey = True, figsize = (12,6))
cc=ax[0,0].contourf(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,CT_ctd[:,ist],np.arange(1,19,1), cmap = cmocean.cm.thermal)
plt.colorbar(cc, ax = ax[0,0])
cd=ax[0,0].contour(dsctd["longitude"][0,3:], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,3:],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,0].set_title("$\\Theta$ ($^{\\circ}$C)")

ax[0,0].set_ylim((6010**b,0))
ax[0,0].set_yticks(zlabels**b)
ax[0,0].set_yticklabels(zlabels)
ax[0,0].set_ylabel("Press (dbar)")

cc=ax[0,1].contourf(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,SA_ctd[:,ist],np.arange(34.9,36.55,0.1), cmap = cmocean.cm.haline)
plt.colorbar(cc, ax = ax[0,1])
cd=ax[0,1].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,1].set_title("$S_A$ (g kg$^{-1}$)")

ax[0,1].set_ylim((6010**b,0))
ax[0,1].set_yticks(zlabels**b)
ax[0,1].set_yticklabels(zlabels)



cc=ax[1,0].pcolor(lon_vmp, pres_vmp**b,np.log10(chi_smooth), vmin = -12, vmax = -7)
plt.colorbar(cc, ax = ax[1,0])
cd=ax[1,0].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,0].set_title("$\\chi_{\\Theta}$ (K$^2$ s$^{-1}$)")

ax[1,0].set_ylim((6010**b,0))
ax[1,0].set_yticks(zlabels**b)
ax[1,0].set_yticklabels(zlabels)
ax[1,0].set_ylabel("Press (dbar)")

cc=ax[1,1].pcolor(lon_vmp, pres_vmp**b,np.log10(chiS_smooth), vmin = -13, vmax = -8)
plt.colorbar(cc, ax = ax[1,1])
cd=ax[1,1].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,1].set_title("$\\chi_{S}$ ((g/kg)$^2$ s$^{-1}$)")

ax[1,1].set_ylim((6010**b,0))
ax[1,1].set_yticks(zlabels**b)
ax[1,1].set_yticklabels(zlabels)

ax[1,0].set_xlabel("Longitude")
ax[1,1].set_xlabel("Longitude")

for ax0 in ax.ravel():
    ax0.fill_between(dsctd["longitude"][0,ist], y1 = 6010**b, y2 = bottom[ist]**b, color = "gray")
    ax0.set_xlim((np.min(dsctd["longitude"][0,ist]), np.max(dsctd["longitude"][0,ist])))
    ax0.annotate("Greenland", xy = (0.01,0.02), xycoords = "axes fraction", ha = "left")
    ax0.annotate("Portugal", xy = (0.99,0.02), xycoords = "axes fraction", ha = "right")
fig.tight_layout()
fig.savefig("mixing_sections_01.png", dpi = 300, bbox_inches = "tight")

In [ ]:
import matplotlib.cm as cm
b= 0.7
zlabels = np.array([0, 500, 1500, 3000, 6000])

bottom = np.full(dsctd["longitude"][0,:].size, np.nan)
for i in range(dsctd["longitude"][0,:].size):
    ij = np.where(np.isfinite(dsctd["temperature"][:,i]))[0][-1]
    bottom[i] = dsctd["pressure"][0,ij]
print(bottom)

    
fig, ax = plt.subplots(2,2,sharex = True, sharey = True, figsize = (12,6))

cc=ax[0,0].pcolor(lon_vmp, pres_vmp**b,np.log10(chi_smooth), vmin = -12, vmax = -7)
plt.colorbar(cc, ax = ax[0,0])
cd=ax[0,0].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,0].set_title("$\\chi_{\\Theta}$ (K$^2$ s$^{-1}$)")

ax[0,0].set_ylim((6010**b,0))
ax[0,0].set_yticks(zlabels**b)
ax[0,0].set_yticklabels(zlabels)
ax[0,0].set_ylabel("Press (dbar)")

cc=ax[0,1].pcolor(lon_vmp, pres_vmp**b,np.log10(chiS_smooth), vmin = -13, vmax = -8)
plt.colorbar(cc, ax = ax[0,1])
cd=ax[0,1].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,1].set_title("$\\chi_{S}$ ((g/kg)$^2$ s$^{-1}$)")

ax[0,1].set_ylim((6010**b,0))
ax[0,1].set_yticks(zlabels**b)
ax[0,1].set_yticklabels(zlabels)



cc=ax[1,0].pcolor(lon_vmp, pres_vmp**b,100*(1-Pdia_smooth/chi_smooth), vmin = 0, vmax = 100, cmap =cm.PuOr_r)
plt.colorbar(cc, ax = ax[1,0])
cd=ax[1,0].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,0].set_title("Isopycnal contribution to thermal mixing (%)")

ax[1,0].set_ylim((6010**b,0))
ax[1,0].set_yticks(zlabels**b)
ax[1,0].set_yticklabels(zlabels)
ax[1,0].set_ylabel("Press (dbar)")

cc=ax[1,1].pcolor(lon_vmp, pres_vmp**b,100*(1-PdiaS_smooth/chiS_smooth), vmin = 0, vmax = 100, cmap =cm.PuOr_r)
plt.colorbar(cc, ax = ax[1,1])
cd=ax[1,1].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,1].set_title("Isopycnal contribution to haline mixing (%)")

ax[1,1].set_ylim((6010**b,0))
ax[1,1].set_yticks(zlabels**b)
ax[1,1].set_yticklabels(zlabels)

ax[1,0].set_xlabel("Longitude")
ax[1,1].set_xlabel("Longitude")

for ax0 in ax.ravel():
    ax0.fill_between(dsctd["longitude"][0,ist], y1 = 6010**b, y2 = bottom[ist]**b, color = "gray")
    ax0.set_xlim((np.min(dsctd["longitude"][0,ist]), np.max(dsctd["longitude"][0,ist])))
    ax0.annotate("Greenland", xy = (0.01,0.02), xycoords = "axes fraction", ha = "left")
    ax0.annotate("Portugal", xy = (0.99,0.02), xycoords = "axes fraction", ha = "right")
fig.tight_layout()
fig.savefig("mixing_sections_02.png", dpi = 300, bbox_inches = "tight")

In [ ]:

import scipy.interpolate as intrp
import matplotlib.cm as cm

chi_clim = np.full(chi.shape, np.nan)
Pdia_clim = np.full(chi.shape, np.nan)
chiS_clim = np.full(chi.shape, np.nan)
PdiaS_clim = np.full(chi.shape, np.nan)
gamman_clim = np.full(chi.shape, np.nan)
for i in range(chi.shape[1]):
    yy = dschi.chiCT_dia.values[:,i] + dschi.chiCT_iso.values[:,i]
    yy1 = dschi.chiSA_dia.values[:,i] + dschi.chiSA_iso.values[:,i]
    xx = dschi.z.values[:,i]
    hh = dschi.h.values[:,i]
    gm = dschi.SG.values
    #print(xx)
    ii = np.isfinite(yy)
    if np.sum(ii)==0:
        continue
    #print(np.max(xx[ii]))
    for j in range(xx.size):
        
        #jj = np.where(np.abs(pres_vmp-xx[j])<=hh[j]/2)[0]
        jj = np.where(np.abs(pres_vmp-xx[j])<=hh[j])[0]
        #if np.sum(jj)>0:
        #    print(yy[j])
        chi_clim[jj,i] = yy[j]
        chiS_clim[jj,i] = yy1[j]
        gamman_clim[jj,i] = gm[j]
        #print(chi_clim[jj,i])
    
    #intchi = intrp.interp1d(xx[ii], yy[ii], bounds_error = False)
    #chi_clim[:,i] = intchi(pres_vmp)
    
    yy = dschi.chiCT_dia.values[:,i]
    yy1 = dschi.chiSA_dia.values[:,i]
    #print(xx)
    ii = np.isfinite(yy)
    if np.sum(ii)==0:
        continue
    print(np.max(xx[ii]))
    #intchi = intrp.interp1d(xx[ii], yy[ii], bounds_error = False)
    #Pdia_clim[:,i] = intchi(pres_vmp)
    for j in range(xx.size):
        #jj = np.where(np.abs(pres_vmp-xx[j])<=hh[j]/2)[0]
        jj = np.where(np.abs(pres_vmp-xx[j])<=hh[j])[0]
        Pdia_clim[jj,i] = yy[j]
        PdiaS_clim[jj,i] = yy1[j]
    




In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature




fig, ax = plt.subplots(2,2,sharex = True, sharey = True, figsize = (12,6))

cc=ax[0,0].pcolor(lon_vmp, pres_vmp**b,np.log10(chi_smooth), vmin = -11, vmax = -7, cmap = cm.inferno)
plt.colorbar(cc, ax = ax[0,0])
cd=ax[0,0].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,0].set_title("VMP $\\chi_{\\Theta}$ (K$^2$ s$^{-1}$)")

ax[0,0].set_ylim((6010**b,0))
ax[0,0].set_yticks(zlabels**b)
ax[0,0].set_yticklabels(zlabels)
ax[0,0].set_ylabel("Press (dbar)")

cc=ax[0,1].pcolor(lon_vmp, pres_vmp**b,np.log10(chi_clim), vmin = -11, vmax = -7, cmap = cm.inferno)
plt.colorbar(cc, ax = ax[0,1])
cd=ax[0,1].contour(lon_vmp, pres_vmp**b,gamman_clim,\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,1].set_title("Climatology $\\chi_{\\Theta}$ (K$^2$ s$^{-1}$)")


ax[0,1].set_ylim((6010**b,0))
ax[0,1].set_yticks(zlabels**b)
ax[0,1].set_yticklabels(zlabels)



cc=ax[1,0].pcolor(lon_vmp, pres_vmp**b,100*(1-Pdia_smooth/chi_smooth), vmin = 0, vmax = 100, cmap =cm.BrBG_r)
plt.colorbar(cc, ax = ax[1,0])
cd=ax[1,0].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,0].set_title("VMP $\\chi_{\\Theta}^{\\rm iso}/ \\chi_{\\Theta}$ (%)")

ax[1,0].set_ylim((6010**b,0))
ax[1,0].set_yticks(zlabels**b)
ax[1,0].set_yticklabels(zlabels)
ax[1,0].set_ylabel("Press (dbar)")

cc=ax[1,1].pcolor(lon_vmp, pres_vmp**b,100*(1-Pdia_clim/chi_clim), vmin = 0, vmax = 100, cmap =cm.BrBG_r)
plt.colorbar(cc, ax = ax[1,1])
cd=ax[1,1].contour(lon_vmp, pres_vmp**b,gamman_clim,\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,1].set_title("Clim. $\\chi_{\\Theta}^{\\rm iso}/ \\chi_{\\Theta}$ (%)")

ax[1,1].set_ylim((6010**b,0))
ax[1,1].set_yticks(zlabels**b)
ax[1,1].set_yticklabels(zlabels)

ax[1,0].set_xlabel("Longitude")
ax[1,1].set_xlabel("Longitude")

for ax0 in ax.ravel():
    ax0.fill_between(dsctd["longitude"][0,ist], y1 = 6010**b, y2 = bottom[ist]**b, color = "gray")
    ax0.set_xlim((np.min(dsctd["longitude"][0,ist]), np.max(dsctd["longitude"][0,ist])))
    ax0.annotate("Greenland", xy = (0.01,0.02), xycoords = "axes fraction", ha = "left")
    ax0.annotate("Portugal", xy = (0.99,0.02), xycoords = "axes fraction", ha = "right")
fig.tight_layout()

ax_map = fig.add_axes([0.08,0.52,0.23,0.23],projection=ccrs.Orthographic(-30, 10))

ax_map.stock_img()
#ax_map.set_global()
ax_map.plot(lon_vmp, lat_vmp,"r.",ms = 0.5, transform = ccrs.PlateCarree())

ax[0,0].annotate("A.", xy = (0.02,0.92), xycoords = "axes fraction", ha = "left", fontweight = "bold")
ax[0,1].annotate("B.", xy = (0.02,0.92), xycoords = "axes fraction", ha = "left", fontweight = "bold")
ax[1,0].annotate("C.", xy = (0.02,0.92), xycoords = "axes fraction", ha = "left", fontweight = "bold")
ax[1,1].annotate("D.", xy = (0.02,0.92), xycoords = "axes fraction", ha = "left", fontweight = "bold")

fig.savefig("OVIDE_temperature_mixing_sections_with_clim.png", dpi = 300, bbox_inches = "tight")

In [ ]:
fig, ax = plt.subplots(2,2,sharex = True, sharey = True, figsize = (12,6))

cc=ax[0,0].pcolor(lon_vmp, pres_vmp**b,np.log10(chiS_smooth), vmin = -13, vmax = -8)
plt.colorbar(cc, ax = ax[0,0])
cd=ax[0,0].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,0].set_title("VMP $\\chi_{S}$ (g kg$^{-2}$ s$^{-1}$)")

ax[0,0].set_ylim((6010**b,0))
ax[0,0].set_yticks(zlabels**b)
ax[0,0].set_yticklabels(zlabels)
ax[0,0].set_ylabel("Press (dbar)")

cc=ax[0,1].pcolor(lon_vmp, pres_vmp**b,np.log10(chiS_clim), vmin = -13, vmax = -8)
plt.colorbar(cc, ax = ax[0,1])
cd=ax[0,1].contour(lon_vmp, pres_vmp**b,gamman_clim,\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[0,1].set_title("Climatology $\\chi_{S}$ (g kg$^{-2}$ s$^{-1}$)")


ax[0,1].set_ylim((6010**b,0))
ax[0,1].set_yticks(zlabels**b)
ax[0,1].set_yticklabels(zlabels)



cc=ax[1,0].pcolor(lon_vmp, pres_vmp**b,100*(1-PdiaS_smooth/chiS_smooth), vmin = 0, vmax = 100, cmap =cm.PuOr_r)
plt.colorbar(cc, ax = ax[1,0])
cd=ax[1,0].contour(dsctd["longitude"][0,ist], dsctd["pressure"][0,:]**b,dsctd["gamman"][:,ist],\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,0].set_title("VMP Isopycnal contribution to haline mixing (%)")

ax[1,0].set_ylim((6010**b,0))
ax[1,0].set_yticks(zlabels**b)
ax[1,0].set_yticklabels(zlabels)
ax[1,0].set_ylabel("Press (dbar)")

cc=ax[1,1].pcolor(lon_vmp, pres_vmp**b,100*(1-PdiaS_clim/chiS_clim), vmin = 0, vmax = 100, cmap =cm.PuOr_r)
plt.colorbar(cc, ax = ax[1,1])
cd=ax[1,1].contour(lon_vmp, pres_vmp**b,gamman_clim,\
                   np.arange(27,28.4,0.2),colors ="w", linewidths = 1)
plt.clabel(cd,fmt = "%1.1f", fontsize = 9)
ax[1,1].set_title("CLIM Isopycnal contribution to haline mixing (%)")

ax[1,1].set_ylim((6010**b,0))
ax[1,1].set_yticks(zlabels**b)
ax[1,1].set_yticklabels(zlabels)

ax[1,0].set_xlabel("Longitude")
ax[1,1].set_xlabel("Longitude")

for ax0 in ax.ravel():
    ax0.fill_between(dsctd["longitude"][0,ist], y1 = 6010**b, y2 = bottom[ist]**b, color = "gray")
    ax0.set_xlim((np.min(dsctd["longitude"][0,ist]), np.max(dsctd["longitude"][0,ist])))
    ax0.annotate("Greenland", xy = (0.01,0.02), xycoords = "axes fraction", ha = "left")
    ax0.annotate("Portugal", xy = (0.99,0.02), xycoords = "axes fraction", ha = "right")
fig.tight_layout()
fig.savefig("salinity_mixing_sections_with_clim.png", dpi = 300, bbox_inches = "tight")

# Water mass transformation in T space

In [ ]:
import scipy.interpolate as intrp




#Airm = 3.72e12#6.12e11
#Airm =6.12e11

DX = (np.max(lon_vmp) - np.min(lon_vmp))*np.cos( np.nanmean(lat_vmp)*180/3.1416 )*110000.
DY = (np.max(lat_vmp) - np.min(lat_vmp))*110000.
Airm = DX*DY
print(Airm*1e-12)

Nsmooth =2
dT = 0.5
Tbin = np.arange(-2,14+dT,dT)


chi_sel = chi[25:,:]
Pdia_sel  = Pdia[25:,:]
grT_sel = grT_mean[25:,:]
CT_sel = theta[25:,:]
depth_sel = depth[25:,:]

chi_Tbins = np.full(Tbin.size, 0.)
Pdia_Tbins = np.full(Tbin.size, 0.)
M_Tbins = np.full(Tbin.size, 0.)
Mdia_Tbins = np.full(Tbin.size, 0.)
grT_Tbins = np.full(Tbin.size, 0.)
Tflux_Tbins = np.full(Tbin.size, 0.)
Tfluxdia_Tbins = np.full(Tbin.size, 0.)

h_Tbins = np.full(Tbin.size, np.nan)
#h_Tbins = np.full(Tbin.size, np.nan)
for i in range(Tbin.size):
    ii = np.abs(CT_sel-Tbin[i])<=dT*(Nsmooth/2)

    xx = chi_sel[ii]
    xx=xx[np.isfinite(xx)]
    if xx.size>0:
        xx=xx[xx<np.quantile(xx,1.)]
        chi_Tbins[i] = np.nanmean(xx)
        M_Tbins[i] = np.nansum(xx*(pres_vmp[1]-pres_vmp[0]))
    #chi_Tbins[i] = np.nanmean(chi_sel[ii])
    Pdia_Tbins[i] = np.nanmean(Pdia_sel[ii])
    Mdia_Tbins[i] = np.nansum(Pdia_sel[ii]*(pres_vmp[1]-pres_vmp[0]))
    grT_Tbins[i] = np.nanmean(np.abs(grT_sel[ii]))
    h_Tbins[i] = np.nansum(np.isfinite(xx))*(pres_vmp[1]-pres_vmp[0])
    

M_Tbins/= chi.shape[1]   
Mdia_Tbins/= chi.shape[1]

ii = np.isfinite(M_Tbins)
Tflux_Tbins[ii] = -0.5*M_Tbins[ii]/dT/Nsmooth
Tfluxdia_Tbins[ii] = -0.5*Mdia_Tbins[ii]/dT/Nsmooth

smooth_Tflux_Tbins = moving_average(Tflux_Tbins,4)
smooth_Tfluxdia_Tbins = moving_average(Tfluxdia_Tbins,4)

Ttrans_Tbins = -np.diff(smooth_Tflux_Tbins)*Airm/dT*1e-6
Ttransdia_Tbins = -np.diff(smooth_Tfluxdia_Tbins)*Airm/dT*1e-6

Tform_Tbins = moving_average(-np.diff(Ttrans_Tbins),4)
Tformdia_Tbins = moving_average(-np.diff(Ttransdia_Tbins),4)


TbinC = 0.5*(Tbin[1:]+Tbin[:-1])
TbinCC =  0.5*(TbinC[1:]+TbinC[:-1])

TbinCC_high = np.arange(-2,14+dT/10,dT/10)
inttform = intrp.interp1d(TbinCC,Tform_Tbins, bounds_error = False)
Tform_high = inttform(TbinCC_high)


fig, ax = plt.subplots(1,3, sharey = True, figsize = (9,4))
l1,=ax[0].plot(smooth_Tflux_Tbins*4000*1000*Airm*1e-12, Tbin,"-", color = "k",label = "total")
ax[0].plot(Tflux_Tbins*4000*1000*Airm*1e-12, Tbin,".", color = l1.get_color())
l2,=ax[0].plot(smooth_Tfluxdia_Tbins*4000*1000*Airm*1e-12, Tbin,"-", label = "diapycnal", color = "darkgreen")
ax[0].plot(Tfluxdia_Tbins*4000*1000*Airm*1e-12, Tbin,".", color = l2.get_color())
ax[0].axvline(0, color = "gray", lw = 1)
ax[0].set_xlabel("Diathermal heat flux (TW)")
ax[0].set_ylabel("$\\Theta$ ($^{\\circ}$C)")
ax[0].legend()

ax[1].plot(Ttrans_Tbins, TbinC,"k.-", label = "isopycnal")
ax[1].plot(Ttransdia_Tbins, TbinC,".-", label = "diapycnal", color = "darkgreen")
ax[1].axvline(0, color = "gray", lw = 1)
ax[1].set_xlabel("Transformation rate (Sv)")
ax[1].set_xlim((-2.2,2.2))

ax[2].plot(Tform_Tbins, TbinCC,"k.-", label = "total")
ax[2].barh(TbinCC[Tform_Tbins>0], Tform_Tbins[Tform_Tbins>0], color = "red", ec = "k", height = 0.5)#, where = Tform_Tbins>0, color = "red", alpha = 0.2)
ax[2].barh(TbinCC[Tform_Tbins<=0], Tform_Tbins[Tform_Tbins<=0], color = "skyblue", ec = "k", height = 0.5)
#ax[2].plot(Tformdia_Tbins,  TbinCC,".-", label = "diapycnal", color = "darkgreen")
#ax[2].fill_betweenx(TbinCC_high, Tform_high, where = Tform_high>0, color = "red", alpha = 0.2)
#ax[2].fill_betweenx(TbinCC_high, Tform_high, where = Tform_high<=0, color = "blue", alpha = 0.2)
ax[2].axvline(0, color = "gray", lw = 1)
ax[2].set_xlim((-0.7,0.7))
ax[2].set_xlabel("Formation rate (Sv)")

for ax0 in ax:
    ax0.grid(True)
fig.tight_layout()


imax = np.argmax(Ttrans_Tbins)
imin = np.argmin(Ttrans_Tbins)
Ov_T = Ttrans_Tbins[imax] - Ttrans_Tbins[imin]

ax[2].annotate("-%1.1f Sv"%(Ttrans_Tbins[imax]), xy = (0.1, 2), ha = "left", fontsize = 12,\
               fontweight = "bold", color = "darkblue")
ax[2].annotate("+%1.1f Sv"%(Ov_T), xy = (-0.4, 8), ha = "left", va = "center", fontsize = 12,\
               fontweight = "bold", color = "darkred")
ax[2].annotate("%1.1f Sv"%(Ttrans_Tbins[imin]), xy = (0.1, 12), ha = "left", va = "center", fontsize = 12,\
               fontweight = "bold", color = "darkblue")

fig.savefig("diathermal_fluxes.png", dpi = 300)

In [ ]:
plt.plot(chi_Tbins, Tbin)
plt.plot(Pdia_Tbins, Tbin)

# Water mass transformation in Sspace

In [ ]:



dS = 0.1
Nsmooth = 2
Sbin = np.arange(34.5,36.7+dS,dS)


chiS_sel = chiS[25:,:]
PdiaS_sel  = PdiaS[25:,:]
grS_sel = np.abs(grS_mean[25:,:])
SA_sel = SA[25:,:]
depth_sel = depth[25:,:]

chiS_Sbins = np.full(Sbin.size, 0.)
PdiaS_Sbins = np.full(Sbin.size, 0.)
MS_Sbins = np.full(Sbin.size, 0.)
MdiaS_Sbins = np.full(Sbin.size, 0.)
grS_Sbins = np.full(Sbin.size, 0.)
Sflux_Sbins = np.full(Sbin.size, 0.)
Sfluxdia_Sbins = np.full(Sbin.size, 0.)
h_Sbins = np.full(Sbin.size, 0.)

for i in range(Sbin.size):
    ii = np.abs(SA_sel-Sbin[i])<=dS*(Nsmooth/2)
    xx = chiS_sel[ii]
    xx=xx[np.isfinite(xx)]
    if xx.size>0:
        xx=xx[xx<np.quantile(xx,0.99)]
        chiS_Sbins[i] = np.nanmean(xx)
        MS_Sbins[i] = np.nansum(xx*(pres_vmp[1]-pres_vmp[0]))
    #chi_Tbins[i] = np.nanmean(chi_sel[ii])
    
    xx = PdiaS_sel[ii]
    xx=xx[np.isfinite(xx)]
    if xx.size>0:
        xx=xx[xx<np.quantile(xx,0.99)]
        PdiaS_Sbins[i] = np.nanmean(xx)
        MdiaS_Sbins[i] = np.nansum(xx*(pres_vmp[1]-pres_vmp[0]))
    grS_Sbins[i] = np.nanmean(np.abs(grT_sel[ii]))
    h_Sbins[i] = np.nansum(np.isfinite(xx))*(pres_vmp[1]-pres_vmp[0])
    

MS_Sbins/= chiS.shape[1]   
MdiaS_Sbins/= chiS.shape[1]

ii = np.isfinite(MS_Sbins)
Sflux_Sbins[ii] = -0.5*MS_Sbins[ii]/dS/Nsmooth
Sfluxdia_Sbins[ii] = -0.5*MdiaS_Sbins[ii]/dS/Nsmooth
smooth_Sflux_Sbins = moving_average(Sflux_Sbins,2)
smooth_Sfluxdia_Sbins = moving_average(Sfluxdia_Sbins,2)

Strans_Sbins = -np.diff(smooth_Sflux_Sbins)*Airm/dS*1e-6
Stransdia_Sbins = -np.diff(smooth_Sfluxdia_Sbins)*Airm/dS*1e-6

Sform_Sbins = moving_average(-np.diff(Strans_Sbins),2)
Sformdia_Sbins = moving_average(-np.diff(Stransdia_Sbins),2)


SbinC = 0.5*(Sbin[1:]+Sbin[:-1])
SbinCC =  0.5*(SbinC[1:]+SbinC[:-1])

SbinCC_high = np.arange(34.5,36.7+dS/10,dS/10)
inttform = intrp.interp1d(SbinCC,Sform_Sbins, bounds_error = False)
Sform_high = inttform(SbinCC_high)

fig, ax = plt.subplots(1,3, sharey = True, figsize = (9,3.5))
l1,=ax[0].plot(smooth_Sflux_Sbins*Airm*1e-6*1027/1000, Sbin,"k-", label = "Total", lw = 2)
ax[0].plot(Sflux_Sbins*Airm*1e-6*1027/1000, Sbin,".", color = l1.get_color())
l2,=ax[0].plot(smooth_Sfluxdia_Sbins*Airm*1e-6*1027/1000, Sbin,ls = "dotted", label = "Diapycnal", color = "darkgreen", lw = 2)
ax[0].plot(Sfluxdia_Sbins*Airm*1e-6*1027/1000, Sbin,".", color = l2.get_color())

l2,=ax[0].plot((smooth_Sflux_Sbins-smooth_Sfluxdia_Sbins)*Airm*1e-6, Sbin,"--", label = "Isopycnal", color = "saddlebrown", lw = 2)
ax[0].plot((Sflux_Sbins-Sfluxdia_Sbins)*Airm*1e-6, Sbin,".", color = l2.get_color())

ax[0].axvline(0, color = "gray", lw = 1)
ax[0].set_xlabel("$F_{S_A|S_A}$ ($10^6$ kg s$^{-1}$)")
ax[0].set_ylabel("$S_A$ (g kg$^{-1}$)")
ax[0].legend()
ax[0].set_xlim((-1.35,0.1))

ax[1].plot(Strans_Sbins, SbinC,"k-", label = "total", lw = 2)
ax[1].plot(Stransdia_Sbins,  SbinC, ls="dotted", label = "diapycnal", color = "darkgreen", lw = 2)
ax[1].plot(Strans_Sbins-Stransdia_Sbins,  SbinC,"--", label = "isopycnal", color = "saddlebrown", lw = 2)

ax[1].axvline(0, color = "gray", lw = 1)
ax[1].set_xlabel("$G$ (Sv)")
ax[1].set_xlim((-3.6,3.6))
ax[1].set_xticks((np.arange(-3,4)))

#ax[2].plot(Sform_Sbins, SbinCC,"k.-", label = "total")
#ax[2].fill_betweenx(SbinCC_high, Sform_high, where = Sform_high>0, color = "red", alpha = 0.2)
#ax[2].fill_betweenx(SbinCC_high, Sform_high, where = Sform_high<=0, color = "blue", alpha = 0.2)
ax[2].barh(SbinCC[Sform_Sbins>0], Sform_Sbins[Sform_Sbins>0], color = "red", ec = "k", height = 0.1)#, where = Tform_Tbins>0, color = "red", alpha = 0.2)
ax[2].barh(SbinCC[Sform_Sbins<=0], Sform_Sbins[Sform_Sbins<=0], color = "skyblue", ec = "k", height = 0.1)
ax[2].plot(Sformdia_Sbins,  SbinCC,ls= "dotted", label = "diapycnal", color = "darkgreen", lw = 2)
ax[2].plot(Sform_Sbins-Sformdia_Sbins,  SbinCC,"--", label = "isopycnal", color = "saddlebrown", lw = 2)

ax[2].axvline(0, color = "gray", lw = 1)
ax[2].set_xlabel("$\\Delta G$ (Sv)")
ax[2].set_xlim((-1.5,1.5))
for ax0 in ax:
    ax0.grid(True)
    
fig.tight_layout()


imax = np.argmax(Strans_Sbins)
imin = np.argmin(Strans_Sbins)

Ov_S = Strans_Sbins[imax] - Strans_Sbins[imin]

ax[2].annotate("-%1.1f Sv"%(Strans_Sbins[imax]), xy = (0.2, 34.75), ha = "left", fontsize = 12,\
               fontweight = "bold", color = "darkblue")
ax[2].annotate("+%1.1f Sv"%(Ov_S), xy = (-1.2, 35.5), ha = "left", va = "center", fontsize = 12,\
               fontweight = "bold", color = "darkred")
ax[2].annotate("%1.1f Sv"%(Strans_Sbins[imin]), xy = (0.2, 36.2), ha = "left", va = "center", fontsize = 12,\
               fontweight = "bold", color = "darkblue")

ax[2].fill_between(x = [-1.5,1.5],y1 = SbinC[imin], y2 = SbinC[imax], zorder = -1, color = "gray", alpha = 0.3)

print(SbinC[imax])
print(SbinC[imin])
print(np.min(Strans_Sbins))
#print( np.sum(Sform_Sbins[Sform_Sbins>0]))

ax[1].set_title("OVIDE (North Atlantic) microstructure observations")

ax[0].annotate("A.", xy = (0.02,0.02), xycoords = "axes fraction", ha = "left", fontweight = "bold")
ax[1].annotate("B.", xy = (0.02,0.02), xycoords = "axes fraction", ha = "left", fontweight = "bold")
ax[2].annotate("C.", xy = (0.02,0.02), xycoords = "axes fraction", ha = "left", fontweight = "bold")

fig.savefig("OVIDE_diahaline_fluxes.png", dpi = 300, bbox_inches = "tight")
#print(np.sum(form_S_int[(Scoords>35.0)&(Scoords<=36)])*1e-6)
#print(np.nansum(form_S_int[(Scoords<=35)])*1e-6)
#print(np.nansum(form_S_int[(Scoords>36)])*1e-6)